# Build Your Own RAG with Groq — Starter Notebook

This Colab notebook is designed for a live workshop.

What you'll build:
1. Upload a PDF
2. Extract text
3. Split it into chunks
4. Create embeddings
5. Store vectors in ChromaDB
6. Retrieve relevant chunks
7. Ask Groq to answer using that context

## 0) Install dependencies
Run this cell first.

In [ ]:
!pip -q install groq pypdf chromadb sentence-transformers

## 1) Imports

In [ ]:
import os
from getpass import getpass

import chromadb
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from groq import Groq

## 2) Set your Groq API key

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

env_path = find_dotenv()
print("Found .env at:", env_path)

load_dotenv(env_path, override=True)

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print("Loaded key exists:", bool(GROQ_API_KEY))

## 3) Choose a Groq model

In [ ]:
GROQ_MODEL = "llama-3.3-70b-versatile"
print("Using model:", GROQ_MODEL)

## 4) Upload a PDF

In [ ]:
from google.colab import files

uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))

In [ ]:
#If uploaded from google
# PDF_PATH = list(uploaded.keys())[0]
PDF_PATH = 'data/sample.pdf'
print("PDF_PATH =", PDF_PATH)

## 5) Extract text from the PDF

In [ ]:
def extract_text_from_pdf(pdf_path: str) -> str:
    reader = PdfReader(pdf_path)
    pages = []

    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:
            pages.append(text)
        else:
            pages.append(f"[No extractable text found on page {i+1}]")

    return "\n\n".join(pages)

raw_text = extract_text_from_pdf(PDF_PATH)
print("Characters extracted:", len(raw_text))
print()
print(raw_text[:1500])

## 6) Chunk the text

You can change:
- `chunk_size`
- `chunk_overlap`

In [ ]:
def chunk_text(text: str, chunk_size: int = 1200, chunk_overlap: int = 200):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - chunk_overlap

    return chunks

chunks = chunk_text(raw_text, chunk_size=1200, chunk_overlap=200)

print("Number of chunks:", len(chunks))
print()
print("First chunk preview:\n")
print(chunks[0][:1000])

## 7) Create embeddings

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def embed_texts(texts):
    return embedding_model.encode(texts, show_progress_bar=True).tolist()

chunk_embeddings = embed_texts(chunks)
print("Created embeddings for", len(chunk_embeddings), "chunks")
print("Embedding dimension:", len(chunk_embeddings[0]))

## 8) Store embeddings in ChromaDB

In [ ]:
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="pdf_rag_demo")

existing = collection.get()
if existing["ids"]:
    collection.delete(ids=existing["ids"])

ids = [f"chunk_{i}" for i in range(len(chunks))]
metadatas = [{"chunk_index": i} for i in range(len(chunks))]

collection.add(
    ids=ids,
    documents=chunks,
    embeddings=chunk_embeddings,
    metadatas=metadatas,
)

print("Stored", len(ids), "chunks in ChromaDB")

## 9) Retrieve relevant chunks for a question

In [ ]:
def retrieve_chunks(question: str, top_k: int = 4):
    question_embedding = embedding_model.encode([question]).tolist()[0]

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k
    )

    retrieved_docs = results["documents"][0]
    retrieved_meta = results["metadatas"][0]
    return retrieved_docs, retrieved_meta

question = "What is this document mainly about?"
retrieved_docs, retrieved_meta = retrieve_chunks(question, top_k=4)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"--- Retrieved Chunk {i} ---")
    print(doc[:800])
    print()

## 10) Build the RAG prompt

In [ ]:
def build_rag_prompt(question: str, retrieved_docs: list[str]) -> str:
    context = "\n\n---\n\n".join(retrieved_docs)

    prompt = f"""
You are a helpful assistant answering questions from a PDF.

Use only the provided context to answer.
If the answer is not in the context, say:
"I could not find the answer in the provided document."

Context:
{context}

Question:
{question}

Answer:
"""
    return prompt.strip()

rag_prompt = build_rag_prompt(question, retrieved_docs)
print(rag_prompt[:3000])

## 11) Ask Groq for the final answer

In [ ]:

client = Groq(api_key=os.environ["GROQ_API_KEY"])
# client = Groq(api_key=GROQ_API_KEY)
def ask_groq(prompt: str, model: str = GROQ_MODEL) -> str:
    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You answer questions using only the provided document context."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.2,
    )
    return completion.choices[0].message.content

final_answer = ask_groq(rag_prompt)
print(final_answer)

## 12) Wrap everything into one function

In [ ]:
def answer_question_with_rag(question: str, top_k: int = 4):
    retrieved_docs, retrieved_meta = retrieve_chunks(question, top_k=top_k)
    prompt = build_rag_prompt(question, retrieved_docs)
    answer = ask_groq(prompt)

    return {
        "question": question,
        "retrieved_docs": retrieved_docs,
        "retrieved_meta": retrieved_meta,
        "answer": answer,
    }

result = answer_question_with_rag("What does this document say about?", top_k=4)
print("QUESTION:\n", result["question"])
print("\nANSWER:\n", result["answer"])

## 13) Inspect the retrieved chunks

In [ ]:
for i, doc in enumerate(result["retrieved_docs"], start=1):
    print(f"===== SOURCE CHUNK {i} =====")
    print(doc[:1200])
    print()

## 14) Try your own question

In [ ]:
my_question = "Replace this with your own question"
# Example:
# my_question = "Summarize the main idea of this document."

if my_question != "Replace this with your own question":
    demo = answer_question_with_rag(my_question, top_k=4)
    print("ANSWER:\n")
    print(demo["answer"])
else:
    print("Edit `my_question` first, then run the cell again.")

## 15) Suggested improvements

After the starter demo works, you can extend it:
- add better chunking
- show sources with page numbers
- tune `top_k`
- improve the prompt
- use metadata filters
- compare different Groq models

## 16) Quick recap

**Load PDF → Chunk text → Create embeddings → Store vectors → Retrieve relevant chunks → Ask LLM**